# ファクターリターンの計算サンプル

`data/raw` の CSV から、NAM ファクターと common ファクター (グループ) の日次リターンを計算します。

1. パラメータを指定する
2. `em_market.raw` でデータを読み込む
3. `em_market.factor` でファクターリターンを計算する
4. `em_market.viz` で図にする (任意)

リターンはすべて % で、累積は日次リターンの単純和です。

## 1. パラメータ

In [ ]:
from_ymd = 20261101  # 計算開始日 (YYYYMMDD)
to_ymd = 20261123  # 計算終了日 (YYYYMMDD)

score_freq = "daily"  # NAM のスコア: "monthly" (直近の月末) / "daily" (当日)

# 図で強調する対象期間 (from_ymd から to_ymd の範囲で指定)
period_from_ymd = 20261117
period_to_ymd = 20261123

# スコアのモニター (IC、TC、リバランス相関)
score_monitor_from_ymd = 20260601  # 表示開始日。IC の 20 日移動平均のため長めに取る
reb_ymd = 20261118  # リバランス日


In [ ]:
%matplotlib inline
# em_market のコードを変更したとき、カーネルを再起動せずに反映する
%load_ext autoreload
%autoreload 2

from pathlib import Path

import pandas as pd
from IPython.display import display

from em_market.factor.common import (
    calc_active_exposures,
    calc_common_factor_returns,
    calc_common_group_exposures,
    calc_common_group_returns,
)
from em_market.factor.nam import calc_nam_factor_returns
from em_market.factor.score import (
    SCORE_COLUMNS,
    calc_ic,
    calc_rebalance_corr,
    calc_tc,
)
from em_market.raw import (
    load_business_days,
    load_common_factors,
    load_returns,
    load_score_dates,
    load_scores,
    load_universe,
    load_weight_history,
    load_weights,
)
from em_market.viz.factor_monitor import (
    NAM_LABELS,
    draw_common_factor_groups,
    draw_nam_factors,
)
from em_market.viz.score_monitor import draw_score_monitor

# notebooks/ とリポジトリ直下のどちらから実行しても data/raw を見つけられるようにする
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW_DIR = ROOT / "data" / "raw"

pd.set_option("display.float_format", "{:.4f}".format)


## 2. データの読み込み

ファイルがない日は警告を出して除外します。

In [ ]:
days = load_business_days(RAW_DIR, from_ymd, to_ymd)

# NAM: 基準日 → 使用するスコア日、スコア、トータルリターン (rtn)、スペシフィックリターン (srtn)
score_dates = load_score_dates(RAW_DIR, from_ymd, to_ymd, score_freq)
scores = load_scores(RAW_DIR, score_dates.unique())
nam_rtn = load_returns(RAW_DIR, score_dates.index, "rtn")
nam_srtn = load_returns(RAW_DIR, score_dates.index, "srtn")

# common: 各日のユニバース、トータルリターン、その日に使う前月のファクター
universe = load_universe(RAW_DIR, days)
returns = load_returns(RAW_DIR, days)
factors = load_common_factors(RAW_DIR, days)

print(f"営業日: {len(days)} 日 ({days[0]} - {days[-1]})")
print(f"NAM のスコア日: {sorted(int(d) for d in score_dates.unique())}")


## 3. NAM ファクター

Region ごとの Q5 (RRI はクラス 1, 2) の等ウェイトリターンから、ユニバースの等ウェイト平均を引いた超過リターンです。トータルリターンとスペシフィックリターンのそれぞれで計算します。

In [ ]:
nam_total = calc_nam_factor_returns(scores, nam_rtn, score_dates)
nam_specific = calc_nam_factor_returns(scores, nam_srtn, score_dates)
nam_total


In [ ]:
# 期間累積 (欠損の日は 0 として足す。期間中に 1 日も値がないファクターは NaN)
pd.DataFrame(
    {
        "total": nam_total.sum(min_count=1),
        "specific": nam_specific.sum(min_count=1),
    }
)


## 4. common ファクター

ファクターごとの単回帰の傾き (スコア 1 標準偏差あたり) と、ファクターグループごとの単純平均です。

In [ ]:
common = calc_common_factor_returns(universe, returns, factors)
groups = calc_common_group_returns(common)
groups


In [ ]:
# 期間累積
groups.sum(min_count=1).sort_values(ascending=False).to_frame("cumulative")


In [ ]:
# 個別ファクターの期間累積 上位・下位 10 (期間中に値がないファクターは除く)
common_cum = common.sum(min_count=1).dropna().sort_values(ascending=False)
pd.concat({"top": common_cum.head(10), "bottom": common_cum.tail(10)}).to_frame("cumulative")


### アクティブエクスポージャー

現行と同じく、対象期間の開始日時点のファンド (port) と BM のウェイトで計算します。各銘柄のエクスポージャーはリターンの回帰と同じ正規スコア (σ 単位) で、アクティブエクスポージャー = Σ (ファンドのウェイト − BM のウェイト) × エクスポージャー です。グループはメンバーの単純平均です。

In [ ]:
exposure_ymd = period_from_ymd  # アクティブエクスポージャーの基準日

exposure_factors = load_common_factors(RAW_DIR, [exposure_ymd])[exposure_ymd]
exposure_universe = pd.Index(load_universe(RAW_DIR, [exposure_ymd])["bid"])
port_weights = load_weights(RAW_DIR, "port", exposure_ymd)
bm_weights = load_weights(RAW_DIR, "bm", exposure_ymd)

active_exposures = calc_active_exposures(
    exposure_factors, exposure_universe, port_weights, bm_weights
)
group_active_exposures = calc_common_group_exposures(active_exposures)
group_active_exposures.sort_values(ascending=False).to_frame("active exposure")


## 5. 図 (任意)

Common factor groups (total return)、NAM factors (total return)、NAM factors (specific return) をそれぞれ 16:9 のスライド全面の大きさの図にします。

| 列 | 内容 |
|---|---|
| Cumulative | `from_ymd` からの累積推移。`period_from_ymd` から `period_to_ymd` の対象期間を網掛け |
| Period | 対象期間の累積推移 |
| Chg Bar | 対象期間の累積リターン (図の中で共通のスケール) |
| Chg % | 対象期間の累積リターン |

線は、Cumulative では累積 0 (点線)、Period では期間の始点 (点線) より上を青、下をえんじ色で描きます。棒と値は、プラスなら青、マイナスならえんじ色です。対象期間に値がないファクターは描きません。

In [ ]:
fig_common = draw_common_factor_groups(
    groups,
    period_from_ymd,
    period_to_ymd,
    history_from_ymd=from_ymd,
    # アクティブエクスポージャーの列を加える場合は、次の 2 行を有効にする
    # active_exposures=group_active_exposures,
    # exposure_ymd=exposure_ymd,
)
fig_common


In [ ]:
fig_nam_total = draw_nam_factors(
    nam_total,
    period_from_ymd,
    period_to_ymd,
    rtn_kind="rtn",
    score_freq=score_freq,
    history_from_ymd=from_ymd,
)
fig_nam_total


In [ ]:
fig_nam_specific = draw_nam_factors(
    nam_specific,
    period_from_ymd,
    period_to_ymd,
    rtn_kind="srtn",
    score_freq=score_freq,
    history_from_ymd=from_ymd,
)
fig_nam_specific


## 6. スコアのモニター (IC、TC、リバランス相関)

スコア (AI、AI70 / BD30、Bigdata) ごとに、`score_monitor_from_ymd` から計算します。相関はいずれも Pearson です。

| 指標 | 内容 |
|---|---|
| IC | 各基準日に使うスコア (NAM と同じく `score_freq` で決まる) と、当日の USD リターンの相関 |
| TC | アクティブウェイト (ファンド − BM) と、同じスコアの相関。アクティブウェイトが 0 の銘柄は除く |
| リバランス相関 | `reb_ymd` 以降で最初の日の日次スコアと、各日の日次スコアの相関 |

In [ ]:
monitor_days = load_business_days(RAW_DIR, score_monitor_from_ymd, to_ymd)
monitor_score_dates = load_score_dates(RAW_DIR, score_monitor_from_ymd, to_ymd, score_freq)
monitor_scores = load_scores(RAW_DIR, monitor_score_dates.unique())
monitor_returns = load_returns(RAW_DIR, monitor_days)
port_history = load_weight_history(RAW_DIR, "port", monitor_days)
bm_history = load_weight_history(RAW_DIR, "bm", monitor_days)

ic = calc_ic(monitor_scores, monitor_returns, monitor_score_dates)
tc = calc_tc(monitor_scores, monitor_score_dates, port_history, bm_history)

# リバランス相関は、score_freq によらず日次スコアで計算する
reb_score_dates = load_score_dates(RAW_DIR, reb_ymd, to_ymd, "daily")
rebalance = calc_rebalance_corr(load_scores(RAW_DIR, reb_score_dates.unique()), reb_ymd)

pd.DataFrame(
    {
        "IC mean (period)": ic.loc[period_from_ymd:period_to_ymd].mean(),
        f"IC mean (since {score_monitor_from_ymd})": ic.mean(),
        "TC (latest)": tc.iloc[-1],
        "rebalance corr (latest)": rebalance.iloc[-1],
    }
).rename(index=NAM_LABELS)


In [ ]:
score_monitor_figs = {}
for column in SCORE_COLUMNS:
    score_monitor_figs[column] = draw_score_monitor(
        ic[column],
        tc[column],
        period_from_ymd,
        period_to_ymd,
        title=f"{NAM_LABELS[column]} score",
        rebalance=rebalance[column],
        history_from_ymd=score_monitor_from_ymd,
    )
    display(score_monitor_figs[column])
